# SHL Grammar Scoring: Whisper encoder embeddings (Kaggle GPU)

Adds one more audio representation to the outputs of `kaggle_transcribe.ipynb`: the hidden states of the **Whisper large-v3 encoder**, averaged over time. The encoder is trained to turn speech into text, so its vectors carry both how the speaker sounds and what they say.

Output: `outputs/emb_audio_whisper.npz`, shape (clips, 9, 1280): every 4th encoder layer (0, 4, ..., 32), stored as float16.

**How to run:** same as `kaggle_transcribe.ipynb` (competition data attached, GPU T4, Internet on, *Save & Run All*). About 15 minutes. Download `emb_audio_whisper.npz` from the *Output* tab into the repo's `outputs/` folder.

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import librosa
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoFeatureExtractor, AutoModel

ENCODER_MODEL = "openai/whisper-large-v3"
LAYERS = list(range(0, 33, 4))  # every 4th of the 33 hidden states, keeps the file small
SAMPLE_RATE = 16_000
WINDOW = 30 * SAMPLE_RATE       # Whisper's encoder always sees 30 s of audio
SAMPLES_PER_FRAME = 320         # one encoder frame = 20 ms

INPUT_DIR = Path("/kaggle/input")
OUT_DIR = Path("/kaggle/working/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "No GPU found: set Accelerator to GPU in the notebook settings"
torch.manual_seed(42)

## Find the clips

Same lookup as `kaggle_transcribe.ipynb`, so rows come out in the same order (train then test, in CSV order).

In [ ]:
def list_files(root, suffixes):
    return sorted(Path(d) / f for d, _, names in os.walk(root, followlinks=True)
                  for f in names if Path(f).suffix.lower() in suffixes)

csv_paths = list_files(INPUT_DIR, {".csv"})
audio_by_stem = {}
for p in list_files(INPUT_DIR, {".wav"}):
    audio_by_stem.setdefault(p.stem, []).append(p)

def find_csv(name):
    matches = [p for p in csv_paths if p.name.lower() == name]
    assert len(matches) == 1, f"expected exactly one {name}, found {matches}"
    return matches[0]

def resolve_audio(value, split):
    candidates = audio_by_stem.get(Path(str(value)).stem, [])
    if len(candidates) > 1:
        candidates = [p for p in candidates
                      if any(split in part.lower() for part in p.relative_to(INPUT_DIR).parts[:-1])]
    assert len(candidates) == 1, f"cannot find a unique audio file for {value!r} ({split}): {candidates}"
    return candidates[0]

files = pd.concat([
    pd.DataFrame({"file_id": pd.read_csv(find_csv("train.csv"))["filename"].astype(str), "split": "train"}),
    pd.DataFrame({"file_id": pd.read_csv(find_csv("test.csv"))["filename"].astype(str), "split": "test"}),
], ignore_index=True)
files["path"] = [resolve_audio(f, s) for f, s in zip(files["file_id"], files["split"])]
print(files["split"].value_counts().to_string())

## Encoder embeddings

Each clip is cut into 30 second windows (Whisper pads the last one with silence). For each kept layer, the frame vectors are averaged over the real audio only, ignoring the padding, and the windows are combined weighted by their length.

In [ ]:
extractor = AutoFeatureExtractor.from_pretrained(ENCODER_MODEL)
encoder = AutoModel.from_pretrained(ENCODER_MODEL).get_encoder().half().to(DEVICE).eval()

pooled_all = []
for path in tqdm(files["path"], desc="Whisper encoder"):
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    windows = [audio[s:s + WINDOW] for s in range(0, len(audio), WINDOW)] or [audio]
    feats = extractor(windows, sampling_rate=SAMPLE_RATE, return_tensors="pt").input_features
    with torch.no_grad():
        hidden = encoder(feats.to(DEVICE, torch.float16), output_hidden_states=True).hidden_states
    n_frames = [max(1, int(np.ceil(len(w) / SAMPLES_PER_FRAME))) for w in windows]
    pooled = [sum(hidden[L][i, :n].float().sum(0) for i, n in enumerate(n_frames)) / sum(n_frames)
              for L in LAYERS]
    pooled_all.append(torch.stack(pooled).cpu().numpy().astype(np.float16))

emb = np.stack(pooled_all)
np.savez_compressed(OUT_DIR / "emb_audio_whisper.npz", file_id=files["file_id"].to_numpy(dtype=str),
                    split=files["split"].to_numpy(dtype=str), emb=emb)
print("emb_audio_whisper.npz", emb.shape, f"{(OUT_DIR / 'emb_audio_whisper.npz').stat().st_size / 1e6:.1f} MB")